<a href="https://colab.research.google.com/github/chiptune234u-lgtm/SandProject/blob/main/sand_PJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
#@title ⏳ 1. 環境設定と素材の準備
#@markdown 生成したい動画の本数や基本設定を入力してください。

NUM_VIDEOS = 1 #@param {type:"slider", min:1, max:50, step:1}
CYCLE_DURATION = "20s" #@param ["20s", "30s"]
FPS = 30
BLOCK_SIZE = 12
WIND_STRENGTH = 0.3

import numpy as np
import cv2
import os
import glob
import random
import colorsys
import time
import gc
from google.colab import drive
from numba import jit
from PIL import Image, ImageDraw, ImageFont

# ドライブマウント
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 共通定数
WIDTH, HEIGHT = 720, 1280
COLS, ROWS = WIDTH // BLOCK_SIZE, HEIGHT // BLOCK_SIZE
DURATION = 30 if CYCLE_DURATION == "30s" else 20

# パス設定
bg_dir = "/content/drive/MyDrive/Colab Notebooks/画像素材"
font_dir = "/content/drive/MyDrive/Colab Notebooks/font_material"
sound_dir = "/content/drive/MyDrive/Colab Notebooks/sound_material"
output_dir = "/content/drive/MyDrive/Colab Notebooks/output"
os.makedirs(output_dir, exist_ok=True)

# 物理演算用（Numba高速化）
@jit(nopython=True)
def apply_wind_numba(v_grid, sand_mask, wind_force):
    rows, cols = v_grid.shape[:2]
    for y in range(rows - 1):
        for x in range(cols):
            if sand_mask[y, x]:
                v_grid[y, x, 0] += wind_force
    return v_grid

print(f"✅ 準備完了: {NUM_VIDEOS}本の生成予約を受け付けました。")

✅ 準備完了: 1本の生成予約を受け付けました。


In [11]:
#@title 🛠️ 2. 生成エンジンの定義

# --- 1. オリジナル物理・爆発関数の定義 ---

def apply_deep_explosion(current_grid, v_grid, power=30):
    active_mask = np.any(current_grid > 0, axis=2)
    active_y, active_x = np.where(active_mask)
    if len(active_y) == 0: return v_grid, (COLS//2, ROWS//2)
    min_x, max_x = np.min(active_x), np.max(active_x)
    explosion_x = random.randint(min_x, max_x)
    min_y, max_y = np.min(active_y), np.max(active_y)
    explosion_y = int(max_y - (max_y - min_y) * 0.2)
    y_indices, x_indices = np.indices((ROWS, COLS))
    dy, dx = y_indices - explosion_y, x_indices - explosion_x
    dist = np.sqrt(dx**2 + dy**2) + 0.1
    explosion_mask = (dist < 60) & active_mask
    if np.any(explosion_mask):
        force = power / dist[explosion_mask]
        v_grid[explosion_mask, 0] += (dx[explosion_mask] / dist[explosion_mask]) * force * 15
        v_grid[explosion_mask, 1] += (dy[explosion_mask] / dist[explosion_mask]) * force * 15
    return v_grid, (explosion_x, explosion_y)

@jit(nopython=True)
def apply_wind_numba(v_grid, sand_mask, wind_force):
    rows, cols = v_grid.shape[:2]
    for y in range(rows - 1):
        for x in range(cols):
            if sand_mask[y, x]:
                v_grid[y, x, 0] += wind_force
    return v_grid

def apply_wind(current_grid, v_grid, wind_force):
    sand_mask = np.any(current_grid > 0, axis=2)
    return apply_wind_numba(v_grid, sand_mask, wind_force)

def update_physics_vectorized(current_grid, v_grid, mode):
    new_grid = np.zeros_like(current_grid)
    new_v_grid = np.zeros_like(v_grid)
    for y in range(ROWS - 1, -1, -1):
        if not np.any(current_grid[y]): continue
        sand_mask_in_row = np.any(current_grid[y, :, :], axis=1)
        sand_x_coords = np.where(sand_mask_in_row)[0]
        colors, velocities = current_grid[y, sand_x_coords], v_grid[y, sand_x_coords]
        vx, vy = velocities[:, 0], velocities[:, 1]
        proposed_nx = np.clip((sand_x_coords + vx).astype(int), 0, COLS - 1)
        proposed_ny = (y + vy + 1).astype(int)
        new_vx, new_vy = vx * 0.9, vy * 0.9
        falling_off_mask = proposed_ny >= ROWS
        if np.any(falling_off_mask):
            target_x_fall = proposed_nx[falling_off_mask]
            new_grid[ROWS-1, target_x_fall] = colors[falling_off_mask]
            new_v_grid[ROWS-1, target_x_fall] = 0.0
        remaining_mask = ~falling_off_mask
        if not np.any(remaining_mask): continue
        sand_x_rem, colors_rem = sand_x_coords[remaining_mask], colors[remaining_mask]
        p_nx, p_ny = proposed_nx[remaining_mask], np.clip(proposed_ny[remaining_mask], 0, ROWS - 1)
        n_vx, n_vy = new_vx[remaining_mask], new_vy[remaining_mask]
        target_indices_linear = p_ny * COLS + p_nx
        unique_indices, first_indices = np.unique(target_indices_linear, return_index=True)
        u_y, u_x = np.divmod(unique_indices, COLS)
        can_move = np.all(new_grid[u_y, u_x] == 0, axis=1)
        valid_indices = first_indices[can_move]
        if len(valid_indices) > 0:
            final_y, final_x = p_ny[valid_indices], p_nx[valid_indices]
            new_grid[final_y, final_x] = colors_rem[valid_indices]
            new_v_grid[final_y, final_x, 0], new_v_grid[final_y, final_x, 1] = n_vx[valid_indices], n_vy[valid_indices]
        mask_moved = np.zeros(len(sand_x_rem), dtype=bool)
        mask_moved[valid_indices] = True
        stuck_indices = np.where(~mask_moved)[0]
        if len(stuck_indices) > 0:
             for i in stuck_indices:
                curr_x, curr_c = sand_x_rem[i], colors_rem[i]
                slide = random.choice([-1, 1])
                tx, ty = curr_x + slide, y + 1
                if 0 <= tx < COLS and ty < ROWS and np.all(new_grid[ty, tx] == 0):
                    new_grid[ty, tx], new_v_grid[ty, tx] = curr_c, [n_vx[i] * 0.5, 0]
                else:
                    new_grid[y, curr_x], new_v_grid[y, curr_x] = curr_c, [0, 0]
    return new_grid, new_v_grid

# --- 2. メイン生成エンジン ---

def generate_one_video(video_idx):
    start_time = time.time() # Start timer
    print(f"\n🎬 {video_idx+1}/{NUM_VIDEOS}本目 生成中...")
    BASE_HUE = random.random()
    bg_img = cv2.cvtColor(cv2.resize(cv2.imread(random.choice(glob.glob(os.path.join(bg_dir, "*")))), (WIDTH, HEIGHT)), cv2.COLOR_BGR2RGB)
    RANDOM_FONT_PATH = random.choice(glob.glob(os.path.join(font_dir, "*.ttf"))) if glob.glob(os.path.join(font_dir, "*.ttf")) else None
    RANDOM_SOUND_PATH = random.choice(glob.glob(os.path.join(sound_dir, "*.*))) if glob.glob(os.path.join(sound_dir, "*.*)) else None

    grid = np.zeros((ROWS, COLS, 3), dtype=np.uint8)
    velocity_grid = np.zeros((ROWS, COLS, 2), dtype=np.float32)

    # 天井砂セット
    fill_limit = int(ROWS * 0.4)
    for y in range(fill_limit):
        width_at_y = int((fill_limit - y) * 1.5)
        start_x, end_x = max(0, (COLS // 2) - width_at_y), min(COLS, (COLS // 2) + width_at_y)
        if end_x > start_x:
            mask = np.random.random(end_x - start_x) > 0.1
            for x_idx in range(start_x, end_x):
                if mask[x_idx - start_x]:
                    r, g, b = colorsys.hsv_to_rgb((BASE_HUE + random.uniform(-0.05, 0.05)) % 1.0, 0.7, 0.9)
                    grid[y, x_idx] = [int(r*255), int(g*255), int(b*255)]

    total_frames = FPS * DURATION
    frames_out = []
    ratio = DURATION / 20.0
    EXPLOSION_FRAMES = [0, int(FPS * 15 * ratio)]
    EMIT_STOP_F = int(FPS * 17 * ratio)
    ROTATION_START_F = total_frames - int(FPS * 2.5)

    # 風設定
    wind_forces = np.zeros(total_frames)
    w_start, w_peak, w_end = int(FPS*7), int(FPS*10), int(FPS*13)
    t_in = np.arange(w_start, w_peak)
    wind_forces[w_start:w_peak] = WIND_STRENGTH * random.choice([-1, 1]) * np.sin((t_in - w_start) / (w_peak - w_start) * np.pi / 2)

    # テキストカラー
    r, g, b = colorsys.hsv_to_rgb((BASE_HUE + 0.5) % 1.0, 0.8, 1.0)
    text_color = (int(r*255), int(g*255), int(b*255))
    try:
        font_l1 = ImageFont.truetype(RANDOM_FONT_PATH, 40)
        font_l2 = ImageFont.truetype(RANDOM_FONT_PATH, 90)
    except:
        font_l1 = font_l2 = ImageFont.load_default()

    # 進捗表示の間隔を設定 (5%ごと)
    progress_interval_frames = max(1, total_frames // 20) # 20 segments for 5% each

    for f in range(total_frames):
        if f in EXPLOSION_FRAMES: velocity_grid, _ = apply_deep_explosion(grid, velocity_grid)
        if f < EMIT_STOP_F:
            for sx in np.random.randint(COLS // 2 - 10, COLS // 2 + 11, 3):
                if not np.any(grid[0, sx]):
                    r, g, b = colorsys.hsv_to_rgb((BASE_HUE + random.uniform(-0.05, 0.05)) % 1.0, 0.7, 0.9)
                    grid[0, sx] = [int(r*255), int(g*255), int(b*255)]

        if abs(wind_forces[f]) > 0.001: velocity_grid = apply_wind(grid, velocity_grid, wind_forces[f])
        grid, velocity_grid = update_physics_vectorized(grid, velocity_grid, "sand")

        sand_layer = cv2.resize(grid, (WIDTH, HEIGHT), interpolation=cv2.INTER_NEAREST)
        if f >= ROTATION_START_F:
            p = (f - ROTATION_START_F) / (total_frames - ROTATION_START_F)
            M = cv2.getRotationMatrix2D((WIDTH // 2, HEIGHT // 2), p * p * (3 - 2 * p) * 180, 1.0)
            sand_layer = cv2.warpAffine(sand_layer, M, (WIDTH, HEIGHT), flags=cv2.INTER_NEAREST)

        frame = bg_img.copy()
        mask_sand = np.any(sand_layer > 0, axis=2)
        frame[mask_sand] = sand_layer[mask_sand]

        if f < ROTATION_START_F:
            next_boom = [ef for ef in EXPLOSION_FRAMES if ef > f]
            if next_boom:
                pil_img = Image.fromarray(frame)
                draw = ImageDraw.Draw(pil_img)
                t1, t2 = "next boom", f"{(next_boom[0]-f)/FPS:.1f}s"
                draw.text((82, int(HEIGHT*0.6)+2), t1, font=font_l1, fill=(0,0,0))
                draw.text((80, int(HEIGHT*0.6)), t1, font=font_l1, fill=text_color)
                draw.text((82, int(HEIGHT*0.6)+97), t2, font=font_l2, fill=(0,0,0))
                draw.text((80, int(HEIGHT*0.6)+95), t2, font=font_l2, fill=text_color)
                frame = np.array(pil_img)

        frames_out.append(frame)

        # 進捗表示
        if f % progress_interval_frames == 0 and f > 0:
            progress_percent = (f / total_frames) * 100
            print(f"  進捗: {progress_percent:.0f}%")

    temp_path = f"/content/temp_{video_idx}.mp4"
    out_v = cv2.VideoWriter(temp_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (WIDTH, HEIGHT))
    for fr in frames_out: out_v.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
    out_v.release()

    final_name = f"sand_{time.strftime('%Y%m%d_%H%M%S')}_{video_idx}.mp4"
    final_path = os.path.join(output_dir, final_name)
    if RANDOM_SOUND_PATH:
        !ffmpeg -y -i "{temp_path}" -i "{RANDOM_SOUND_PATH}" -filter_complex "[1:a]afade=t=in:st=0:d=1,afade=t=out:st={DURATION-1}:d=1[a]" -map 0:v -map "[a]" -c:v libx264 -crf 23 -preset ultrafast -pix_fmt yuv420p -c:a aac -shortest "{final_path}" -loglevel error
    else:
        !ffmpeg -y -i "{temp_path}" -vcodec libx264 -crf 23 -preset ultrafast -pix_fmt yuv420p "{final_path}" -loglevel error

    if os.path.exists(temp_path): os.remove(temp_path)
    del frames_out; gc.collect()
    end_time = time.time() # End timer
    elapsed_time = end_time - start_time
    print(f"✅ 完成: {final_name} (所要時間: {elapsed_time:.2f}秒)")

print("✅ エンジン定義完了（オリジナル物理ロジック移植済）")

SyntaxError: invalid syntax (ipython-input-599133031.py, line 89)

In [ ]:
#@title 🚀 3. 連続生成の開始

for i in range(NUM_VIDEOS):
    generate_one_video(i)

print(f"\n🎉 すべての生成が終了しました（計 {NUM_VIDEOS} 本）")